# Rib Interface
> Find the modes in a rib waveguide

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import meow as mw

## Structures

In [ ]:
wl = 1.0
widths = [0.8, 1.0]
t_slab, t_soi, t_ox = 0.2, 1.5, 0.2
n_Si, n_SiO2 = 3.4, 1.4
w_sim, h_sim, y_bot = 3.0, 2.5, -0.5

si = mw.IndexMaterial(n=n_Si, name="Si", meta={"color": "orange"})
sio2 = mw.IndexMaterial(n=n_SiO2, name="SiO2", meta={"color": "steelblue"})

env = mw.Environment(wl=wl, T=25.0)
mesh = mw.Mesh2D(
    x=np.linspace(-w_sim / 2, w_sim / 2, 101),
    y=np.linspace(y_bot, y_bot + h_sim, 101),
    # num_pml=(10, 10),
    subpixel_smoothing=False,
)


def make_cross_section(w):
    structures = [
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=y_bot,
                y_max=t_slab + t_ox,
            ),
            mesh_order=2,
        ),
        mw.Structure2D(
            material=sio2,
            geometry=mw.Rectangle(
                x_min=-w / 2 - t_ox / 2,
                x_max=w / 2 + t_ox / 2,
                y_min=t_slab + t_ox,
                y_max=t_soi + t_ox,
            ),
            mesh_order=2,
        ),
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w_sim / 2,
                x_max=w_sim / 2,
                y_min=0.0,
                y_max=t_slab,
            ),
            mesh_order=1,
        ),
        mw.Structure2D(
            material=si,
            geometry=mw.Rectangle(
                x_min=-w / 2,
                x_max=w / 2,
                y_min=t_slab,
                y_max=t_soi,
            ),
            mesh_order=1,
        ),
    ]
    return mw.CrossSection(
        structures=structures,
        mesh=mesh,
        env=env,
        subpixel_smoothing=True,
    )


cs_left = make_cross_section(widths[0])
cs_right = make_cross_section(widths[1])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cs_left._visualize(ax=axes[0], show=False)
axes[0].set_title(f"Left (w={widths[0]} um)")
cs_right._visualize(ax=axes[1], show=False)
axes[1].set_title(f"Right (w={widths[1]} um)")
plt.tight_layout()
plt.show()

## Modes

In [ ]:
num_modes = 100
modes_left = mw.compute_modes(cs_left, num_modes=num_modes)
modes_right = mw.compute_modes(cs_right, num_modes=num_modes)

In [ ]:
# Visualize left and right mode sets
print("Left cross-section modes")
mw.visualize(modes_left[:2], fields=("Ex", "Hx"))

print("Right cross-section modes")
mw.visualize(modes_right[:2], fields=("Ex", "Hx"))

## Interface

In [ ]:
S, pm = mw.eme.compute_interface_s_matrix(
    modes_left, modes_right, passivity_method="subtract"
)
S_sub = mw.downselect_s((S, pm), ["left@0", "left@1", "right@0", "right@1"])

In [ ]:
mw.visualize(S_sub)